# Lab 3: SELF-ASSESSMENT: Compiling Models for Dataflow Execution

**Course**: *Dataflow Architectures for AI: From Principles to Practice*  
**Week**: 3, Compilation for Dataflow  
**Instructor**: Dr. Kaoutar El Maghraoui (IBM Research)  
**Lab Instructor**: Dr. Rashed Z. Bhatti (IBM Research)  
**Runtime**: Coursera Labs (CPU only — **no GPU needed here**)  
**Expected time**: 15–20 minutes

---

> **Self-Assessment notebook — the 40 points here are ungraded.** This is where
> you check how well your Lab 3 run went. You do not write code or re-compile
> anything here — you paste the `lab3_results.json` your Colab run produced,
> and all **40 ungraded Self-Assessment points** are checked from it
> automatically. There is no written analysis and no human review.
>
> **The graded item for this lab is the separate Concept Check quiz** (10 points,
> auto-graded in the Coursera Graded Assignment section). It is mandatory and
> requires **80%** to pass. Nothing in *this* notebook affects your course grade.

## Learning objectives
This notebook assesses the objectives of the Colab lab:
1. Compile a model with torch.compile and measure the speedup over eager mode.
2. Ablate the eager / aot_eager / inductor backends to locate where the speedup comes from.
3. Detect, count, and fix graph breaks that prevent whole-graph compilation.
4. Read the generated Triton vs cuBLAS kernels and relate fusion to dataflow.

## Prerequisites
A completed run of
[Lab3_Compiling_for_Dataflow_STUDENT.ipynb](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/gpu-labs/student/Lab3_Compiling_for_Dataflow_STUDENT.ipynb)
on Google Colab (free T4), with **Part 7** run so that **`lab3_results.json`**
exists.

## Why this notebook exists

Lab 3 compiles a real model on a real GPU, so you run it on **Google Colab**
(free T4). This Coursera notebook is where your results get **checked, for your
own feedback**.

The grading environment has **no GPU, no model download, and no network
access**, so it cannot re-compile your model. What it *can* do — and what it
does — is check that the numbers you measured are **internally consistent**:
each speedup has to follow from the two latencies behind it, the compilation
overhead from first-call minus steady-state, the subgraph count from the
graph-break count. Those relationships hold for any real run on any GPU, and
they break when values are edited by hand.

## The workflow

| Step | Where | What you do |
|------|-------|-------------|
| 1 | GitHub → Colab | Open [Lab3_Compiling_for_Dataflow_STUDENT.ipynb](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/gpu-labs/student/Lab3_Compiling_for_Dataflow_STUDENT.ipynb), then **File → Save a copy in Drive** |
| 2 | Colab | Set **Runtime → Change runtime type → T4 GPU**, then complete every TODO and run all cells |
| 3 | Colab | Run **Part 7**. It writes **`lab3_results.json`** |
| 4 | Colab | Copy the JSON it prints, or open the file browser (📁, left sidebar) and double-click `lab3_results.json` |
| 5 | **Here** | Paste it into the `RESULTS_JSON` cell at the top of this notebook, then **Kernel → Restart & Run All** for your self-assessment |

> **The free Colab T4 is all you need.** Every point below is reachable with a
> single free-tier run. You do not need Colab Pro, and you do not need an L4,
> V100 or A100. If Colab happens to assign you a different GPU, your
> self-assessment is unaffected.
>
> **Students are encouraged to experiment with other GPUs and accelerators to
> expand their learning.** Re-running the compilation ablation on an L4 or A100
> is genuinely instructive. There is **no extra credit** for doing so; all 40
> Self-Assessment points come from a single free-tier run.

> **This lab is checked on consistency, never on speed.** `torch.compile` does
> **not** reliably beat eager on a block this small. The reference T4 run
> measured a speedup of just **1.05x**, and on some torch versions `inductor`
> comes out *slower* than the `eager` backend. **No check below requires
> compilation to be faster**, or requires any particular backend to win.

## How this notebook is checked (40 ungraded Self-Assessment points)

| Part | Points | What it checks |
|------|-------:|----------------|
| **A — Structural invariants** | 12 | Facts true of *any* correct run: units, all three backends present, numerics preserved, required fields |
| **B — Self-consistency** | 18 | Your derived values recomputed from your own raw measurements |
| **C — Physics & plausibility** | 10 | Graph-break mechanics, ordering, and wide sanity bands |
| **Ungraded notebook total** | **40** | fully auto-graded from your JSON for Self-Assessment |

**40 points is a full self-assessment score, and a single free-tier T4 run earns
all of it.** Every check is **auto-graded from the JSON you paste** — there is no
written analysis and no human review. No check requires any particular hardware:
a T4, L4, V100 or A100 run all score the same.

A **score table** near the end of this notebook reports your 40-point
self-assessment total, so you know exactly which parts of the lab went well.

**Graded Quiz points:**  
A 10 point graded Coursera Assignment quiz is given to assess your understanding
from this lab experience. It is **mandatory**, and you need **80%** to pass.

| Quiz | Points | Comes from |
|---|---:|---|
| Concept Check quiz | 10 | Auto-graded Coursera Assignment |

> **Rounding is fine.** Every numeric check allows 2% relative tolerance, which
> is far more than the rounding in the lab's printed output. Transfer the values
> as the lab produced them and you will be well inside it.

> **Do not skip the Concept Check quiz.** It lives in the Coursera **Graded
> Assignment** section, is auto-graded as its own item, and is worth **10 graded
> points** — the only points for this lab that count toward your course grade.
> It is mandatory and requires 80% to pass.


## Use your own measurements

These 40 points are ungraded, so there is nothing to gain by editing numbers —
the value of this notebook is the feedback. Use measurements from **your own**
Colab run of Lab 3.

Be aware that the checks in Part B are *cross-checks*: they recompute each
derived quantity from your own raw inputs and require the two to agree. Numbers
that were not produced together by a real run generally fail several of these
at once. Transcription and rounding are fine; invented values are not.


---
## Your results  &nbsp;·&nbsp; *start here*

Paste the **entire contents** of `lab3_results.json` from your Colab run into the
next cell, between the triple quotes — from the opening `{` through the closing `}`.

Keep the raw-string wrapper (the `r` and the triple quotes) exactly as it is: it
means backslashes and quotes inside your JSON cannot break the cell.

```python
RESULTS_JSON = r"""
paste the contents of lab3_results.json here
"""
```

> **This cell only stores your text — it does not check it yet.** After pasting,
> use **Kernel → Restart & Run All**: the Setup cell below loads the helpers, and
> every check in Parts A–C runs against what you pasted here.

> **If you cannot paste**, upload `lab3_results.json` next to this notebook
> instead (Coursera Labs: the **upload** button in the file browser) and leave the
> placeholder text alone. The loader checks the pasted text first, then falls back
> to the file.


In [ ]:
# Paste the contents of lab3_results.json between the triple quotes.
# (Leave the surrounding raw-string triple quotes exactly as they are.)
# This cell only stores the text; the checks run after Kernel -> Restart & Run All.

# your code here
raise NotImplementedError
RESULTS_JSON = r"""
paste the contents of lab3_results.json here
"""


---
## Setup

The next two cells are **read-only — just run them.** The first defines the
loader and the comparison helpers used by every check below; the second loads
the JSON you pasted above and echoes it back so you can confirm it arrived
intact. They import only the Python standard library: nothing here needs
`torch`, a GPU, or the network.


In [ ]:
# === Grading helpers (read-only) =======================================
# Standard library only: nothing here needs torch, a GPU, or the network.
import json, ast, os, math, re

TOL = 0.02            # 2% relative tolerance on derived quantities

# The three backends Part 2 ablates, in increasing stack depth.
BACKENDS = ("eager", "aot_eager", "inductor")

# Numerical-agreement ceiling. The lab's own checkpoints use 1e-2 for the
# backend ablation and 1e-3 for the compiled-vs-eager comparison.
MAX_DIFF_CEIL = 1e-2

REQUIRED_LS_KEYS = {
    "accelerator", "graph_capture", "lowering_pipeline", "codegen",
    "eager_vs_compiled_speedup", "graph_breaks_before_fix",
    "graph_breaks_after_fix",
}

# --- score bookkeeping (for the courtesy readout near the end) ----------
# Each graded cell calls record(...) as its LAST line, after its asserts. A
# cell that raises never reaches that line, so "absent" means "failed".
_SCORE = {}

def record(gid):
    """Mark a graded cell as passed. Called at the end of each graded cell."""
    _SCORE[gid] = True

# Every autograded cell, with its part and points. The tally walks this, so it
# must stay in step with the cells' nbgrader metadata (a test enforces that).
MANIFEST = [
    ("a_perf_units",              "A", 2),
    ("a_three_backends",          "A", 3),
    ("a_numerics_preserved",      "A", 2),
    ("a_landscape_keys",          "A", 2),
    ("a_break_counts_int",        "A", 2),
    ("a_codegen_shape",           "A", 1),
    ("b_speedup_from_latencies",  "B", 4),
    ("b_landscape_speedup",       "B", 2),
    ("b_compile_overhead",        "B", 4),
    ("b_savings_per_call",        "B", 2),
    ("b_slowdown_from_breaks",    "B", 3),
    ("b_fix_speedup",             "B", 2),
    ("b_fixed_matches_clean",     "B", 1),
    ("c_graphs_equal_breaks_plus_one", "C", 3),
    ("c_breaks_detected_then_fixed",   "C", 3),
    ("c_breaks_cost_time",        "C", 2),
    ("c_latency_bands",           "C", 1),
    ("c_break_even_magnitude",    "C", 1),
]
CHECK_LABELS = {
    "a_perf_units":             "Part 1 latencies are positive milliseconds",
    "a_three_backends":         "all three backends reported",
    "a_numerics_preserved":     "compilation preserved numerics",
    "a_landscape_keys":         "landscape_row_w3 has all 7 fields, filled",
    "a_break_counts_int":       "graph-break counts are whole numbers",
    "a_codegen_shape":          "codegen kernel counts well-formed",
    "b_speedup_from_latencies": "speedup == eager_ms / compiled_ms",
    "b_landscape_speedup":      "row speedup mirrors perf['speedup']",
    "b_compile_overhead":       "overhead == first_call - steady_state",
    "b_savings_per_call":       "savings == eager_baseline - steady_state",
    "b_slowdown_from_breaks":   "slowdown == broken_ms / clean_ms",
    "b_fix_speedup":            "fix speedup == broken_ms / fixed_ms",
    "b_fixed_matches_clean":    "fixed model back at clean latency",
    "c_graphs_equal_breaks_plus_one": "subgraphs == breaks + 1 (exact)",
    "c_breaks_detected_then_fixed":   "breaks found, then driven to zero",
    "c_breaks_cost_time":       "graph breaks cost time, not save it",
    "c_latency_bands":          "latencies in a plausible band",
    "c_break_even_magnitude":   "break-even order of magnitude",
}
PART_NAMES = {
    "A": "Structural invariants",
    "B": "Self-consistency",
    "C": "Physics and plausibility",
}


def rel(a, b):
    """Relative difference between a and b, safe at b == 0."""
    return abs(float(a) - float(b)) / max(abs(float(b)), 1e-12)


def num(x, field="value"):
    """Read x as a float, forgiving the ways a real export can mangle one.

    The lab writes plain JSON, but students paste through editors and an older
    version used default=str, so a value can arrive as "0.951", "0.951 ms" or
    "np.float64(0.951)" instead of a number. Coerce rather than fail: a unit
    suffix or a repr wrapper is a formatting accident, not a wrong answer.
    """
    if isinstance(x, bool):
        raise AssertionError(f"{field}: expected a number, got a boolean.")
    if x is None:
        raise AssertionError(f"{field}: is missing (None). Re-run the Colab lab.")
    if isinstance(x, (int, float)):
        return float(x)
    if isinstance(x, str):
        t = x.strip().replace(",", "").replace("_", "")
        if t.lower() in ("inf", "infinity", "+inf"):
            return float("inf")
        # np.float64(0.95) / tensor(0.95) / Decimal('0.95') -> 0.95
        m = re.fullmatch(r"[A-Za-z_][A-Za-z0-9_.]*\(\s*['\"]?([-\d.eE+]+)['\"]?\s*\)", t)
        if m:
            t = m.group(1)
        for suffix in ("iters", "iterations", "ms", "us", "s", "tok/s", "MB",
                       "GB", "x", "×"):
            if t.endswith(suffix):
                t = t[: -len(suffix)].strip()
        try:
            return float(t)
        except ValueError:
            pass
    raise AssertionError(f"{field}: could not read {x!r} as a number.")


def _parse(text):
    text = text.strip()
    if not text:
        raise AssertionError("No results provided.")
    try:
        return json.loads(text)
    except Exception:
        pass
    try:
        return ast.literal_eval(text)
    except Exception:
        pass
    if text.startswith("{") and not text.endswith("}"):
        raise AssertionError(
            "Your pasted JSON looks incomplete. Copy from the opening '{' "
            "through the final '}'.")
    if any(q in text for q in ("“", "”", "‘", "’")):
        raise AssertionError(
            "Your pasted text contains curly 'smart quotes', which are not "
            "valid JSON. Paste as plain text.")
    raise AssertionError(
        "Could not parse your results. Paste the exact contents of "
        "lab3_results.json. (Curly 'smart quotes' from a word processor "
        "will break JSON - paste as plain text.)")


def load_results(blob=""):
    """Pasted JSON first, then an uploaded lab3_results.json, else explain."""
    data, src = None, None
    if blob and blob.strip() and "paste the contents" not in blob:
        data, src = _parse(blob), "pasted text"
    else:
        for p in ("lab3_results.json", "submission/lab3_results.json",
                  os.path.expanduser("~/lab3_results.json")):
            if os.path.exists(p):
                with open(p) as f:
                    data, src = _parse(f.read()), p
                break
    if data is None:
        raise AssertionError(
            "No results found. Either paste the contents of lab3_results.json "
            "into RESULTS_JSON below, or upload the file next to this notebook.")
    if not isinstance(data, dict):
        raise AssertionError("Your results should be a JSON object (a dict).")

    for k in ("perf", "backends", "graph_breaks", "graph_break_fix",
              "landscape_row_w3"):
        if k not in data:
            raise AssertionError(
                f"Key '{k}' is missing. Re-run Part 7 of the Colab lab to "
                f"regenerate lab3_results.json. (If your file has no "
                f"'backends' key you are running an older copy of Lab 3 - "
                f"re-download it.)")

    perf_in = {str(k).strip().lower(): v for k, v in dict(data["perf"]).items()}
    for k in ("eager_ms", "compiled_ms", "speedup"):
        if k not in perf_in:
            raise AssertionError(
                f"perf is missing '{k}'. Did every Part 1 cell of the Colab "
                f"lab run?")
    perf = {k: num(perf_in[k], f"perf['{k}']")
            for k in ("eager_ms", "compiled_ms", "speedup")}
    perf["max_diff"] = (num(perf_in["max_diff"], "perf['max_diff']")
                        if perf_in.get("max_diff") is not None else None)

    backends = {}
    for b, v in dict(data["backends"]).items():
        v = dict(v)
        backends[str(b).strip().lower()] = {
            "latency_ms": num(v.get("latency_ms"), f"backends['{b}']['latency_ms']"),
            "max_diff": (num(v["max_diff"], f"backends['{b}']['max_diff']")
                         if v.get("max_diff") is not None else None),
        }

    gb = dict(data["graph_breaks"])
    graph_breaks = {
        "n_graph_breaks": int(num(gb.get("n_graph_breaks"), "n_graph_breaks")),
        "n_graphs":       int(num(gb.get("n_graphs"), "n_graphs")),
        "broken_ms":      num(gb.get("broken_ms"), "broken_ms"),
        "clean_ms":       num(gb.get("clean_ms"), "clean_ms"),
        "slowdown":       num(gb.get("slowdown"), "slowdown"),
    }

    fx = dict(data["graph_break_fix"])
    fix = {
        "fixed_breaks": int(num(fx.get("fixed_breaks"), "fixed_breaks")),
        "fixed_ms":     num(fx.get("fixed_ms"), "fixed_ms"),
        "fix_diff":     (num(fx["fix_diff"], "fix_diff")
                         if fx.get("fix_diff") is not None else None),
        "speedup_vs_broken": (num(fx["speedup_vs_broken"], "speedup_vs_broken")
                              if fx.get("speedup_vs_broken") is not None else None),
        # Exported since v1.1: the clean model re-measured in the same cell as
        # fixed_ms. Older files lack it, so fall back to graph_breaks.clean_ms.
        "clean_ms_recheck": (num(fx["clean_ms_recheck"], "clean_ms_recheck")
                             if fx.get("clean_ms_recheck") is not None else None),
    }

    cc = dict(data.get("compile_cost") or {})
    compile_cost = {k: (num(cc[k], f"compile_cost['{k}']")
                        if cc.get(k) is not None else None)
                    for k in ("first_call_ms", "steady_state_ms",
                              "eager_baseline_ms", "compilation_overhead_ms",
                              "savings_per_call_ms", "break_even_iters")}

    cg = dict(data.get("codegen") or {})
    codegen = {
        "triton_kernels": (int(num(cg["triton_kernels"], "triton_kernels"))
                           if cg.get("triton_kernels") is not None else None),
        "extern_kernels": (int(num(cg["extern_kernels"], "extern_kernels"))
                           if cg.get("extern_kernels") is not None else None),
        "source": (str(cg["source"]) if cg.get("source") is not None else None),
    }

    return {
        "perf": perf,
        "compile_cost": compile_cost,
        "backends": backends,
        "graph_breaks": graph_breaks,
        "graph_break_fix": fix,
        "codegen": codegen,
        "landscape_row_w3": dict(data["landscape_row_w3"]),
        "environment": dict(data.get("environment") or {}),
        "_source": src,
    }


print(f"Helpers loaded. Tolerance on derived values: {TOL*100:.0f}%")
print(f"Backends ablated in Part 2: {', '.join(BACKENDS)}")
print(f"Self-Assessment checks: {len(MANIFEST)}   "
      f"Parts A-C total: {sum(p for _, _part, p in MANIFEST)} ungraded points"
      f"   (fully auto-graded, no human review)")


In [ ]:
# === Load and echo your results (read-only) ============================
# Every graded cell below calls load_results() itself, so one bad field can
# never cascade into unrelated failures.
_SCORE.clear()   # fresh top-to-bottom run: forget any earlier cell outcomes

_r  = load_results(RESULTS_JSON)
_p  = _r["perf"]
_cc = _r["compile_cost"]
_gb = _r["graph_breaks"]
_fx = _r["graph_break_fix"]
_cg = _r["codegen"]

print(f"Loaded from: {_r['_source']}\n")
print("Part 1 - compile and measure")
print(f"  eager      {_p['eager_ms']:9.3f} ms")
print(f"  compiled   {_p['compiled_ms']:9.3f} ms")
print(f"  speedup    {_p['speedup']:9.3f}x")
if _p["max_diff"] is not None:
    print(f"  max_diff   {_p['max_diff']:9.2e}")

print("\nPart 1b - compilation cost")
for _k in ("first_call_ms", "steady_state_ms", "eager_baseline_ms",
           "compilation_overhead_ms", "savings_per_call_ms"):
    _v = _cc.get(_k)
    print(f"  {_k:24s} {_v:12.3f}" if _v is not None else f"  {_k:24s}  (absent)")
_be = _cc.get("break_even_iters")
print(f"  {'break_even_iters':24s} "
      + ("never (compiled is not faster)" if _be is None else f"{_be:12,.0f}"))

print("\nPart 2 - backend ablation")
for _b in BACKENDS:
    if _b in _r["backends"]:
        print(f"  {_b:12s} {_r['backends'][_b]['latency_ms']:9.3f} ms")
    else:
        print(f"  {_b:12s}   (MISSING)")

print("\nPart 3/4 - graph breaks")
print(f"  breaks {_gb['n_graph_breaks']} -> subgraphs {_gb['n_graphs']}")
print(f"  broken {_gb['broken_ms']:.3f} ms vs clean {_gb['clean_ms']:.3f} ms "
      f"= {_gb['slowdown']:.2f}x slowdown")
print(f"  after the fix: {_fx['fixed_breaks']} breaks, {_fx['fixed_ms']:.3f} ms")
if _fx.get("clean_ms_recheck") is not None:
    print(f"  clean re-measured in the same cell: {_fx['clean_ms_recheck']:.3f} ms")

print("\nPart 5 - generated code")
if _cg["triton_kernels"] is None:
    print("  kernel counts not collected (accepted; no points lost)")
else:
    print(f"  {_cg['triton_kernels']} Triton + {_cg['extern_kernels']} extern "
          f"(source: {_cg['source']})")

print(f"\nPart 6 - {_r['landscape_row_w3'].get('accelerator')}")
print(f"  codegen: {_r['landscape_row_w3'].get('codegen')}")
_env = _r["environment"]
if _env:
    print(f"\nEnvironment: {_env.get('device')}, torch {_env.get('torch_version')}")
print("\nCheck these against your Colab output before continuing.")


---
# Part A — Structural invariants (12 ungraded points)

These hold for **any** correct run on **any** GPU. They check that your results
are complete, in the right units, and describe the experiment the lab actually
ran.

In [ ]:
# A1 (2 pts): the Part 1 latencies must be in MILLISECONDS and positive.
_r = load_results(RESULTS_JSON); _p = _r["perf"]
for _k in ("eager_ms", "compiled_ms"):
    assert _p[_k] > 0, f"perf['{_k}'] must be positive, got {_p[_k]}."
    assert _p[_k] < 10000, (
        f"perf['{_k}'] = {_p[_k]:g} ms is implausibly large for one forward "
        f"pass of a small transformer block. Did you report SECONDS instead of "
        f"milliseconds, or time the first (compiling) call?")
assert _p["speedup"] > 0, "perf['speedup'] must be positive."
print(f"[OK] latencies in ms and positive: eager {_p['eager_ms']:.3f}, "
      f"compiled {_p['compiled_ms']:.3f}")
record("a_perf_units")


In [ ]:
# A2 (3 pts): Part 2 must report all three backends of the ablation.
# The ablation only means something with all three rungs of the ladder:
# eager (Dynamo only), aot_eager (+ AOTAutograd), inductor (+ Triton codegen).
_r = load_results(RESULTS_JSON); _bk = _r["backends"]
_missing = [_b for _b in BACKENDS if _b not in _bk]
assert not _missing, (
    f"backends is missing {_missing}. Part 2 ablates all three: "
    f"{', '.join(BACKENDS)}. If the loop broke early, re-run it.")
for _b in BACKENDS:
    assert _bk[_b]["latency_ms"] > 0, f"backends['{_b}']['latency_ms'] must be positive."
print("[OK] all three backends reported: "
      + ", ".join(f"{_b}={_bk[_b]['latency_ms']:.3f}ms" for _b in BACKENDS))
record("a_three_backends")


In [ ]:
# A3 (2 pts): compilation must preserve numerics.
# This is the invariant that makes a speedup meaningful at all: a "faster"
# model that computes something else is not an optimisation. fp32 accumulation
# order changes, so a small nonzero difference is expected and fine.
_r = load_results(RESULTS_JSON); _p = _r["perf"]; _bk = _r["backends"]
if _p["max_diff"] is not None:
    assert _p["max_diff"] >= 0, "max_diff cannot be negative."
    assert _p["max_diff"] < MAX_DIFF_CEIL, (
        f"perf['max_diff'] = {_p['max_diff']:.2e} exceeds {MAX_DIFF_CEIL:g}. "
        f"The compiled model should match eager to within fp32 reassociation "
        f"noise (the reference run saw about 7e-07).")
for _b, _v in _bk.items():
    if _v["max_diff"] is not None:
        assert _v["max_diff"] < MAX_DIFF_CEIL, (
            f"backends['{_b}']['max_diff'] = {_v['max_diff']:.2e} exceeds "
            f"{MAX_DIFF_CEIL:g}: that backend changed the model's output.")
print("[OK] compilation preserved numerics across every backend reported")
record("a_numerics_preserved")


In [ ]:
# A4 (2 pts): landscape_row_w3 must have exactly the 7 required fields, filled.
_r = load_results(RESULTS_JSON); _ls = _r["landscape_row_w3"]
_got = set(_ls)
assert _got == REQUIRED_LS_KEYS, (
    f"landscape_row_w3 keys are wrong.\n  missing: {sorted(REQUIRED_LS_KEYS - _got)}"
    f"\n  unexpected: {sorted(_got - REQUIRED_LS_KEYS)}")
for _k, _v in _ls.items():
    assert _v is not None, f"landscape_row_w3['{_k}'] is still None."
for _k in ("accelerator", "graph_capture", "lowering_pipeline", "codegen"):
    assert isinstance(_ls[_k], str) and _ls[_k].strip(), (
        f"landscape_row_w3['{_k}'] should be a short non-empty description.")
print("[OK] all 7 landscape_row_w3 fields present and filled")
record("a_landscape_keys")


In [ ]:
# A5 (2 pts): graph-break counts must be non-negative whole numbers.
# dynamo.explain() returns counts, not timings: a fractional or negative value
# means something other than graph_break_count / graph_count was exported.
_r = load_results(RESULTS_JSON); _gb = _r["graph_breaks"]; _fx = _r["graph_break_fix"]
for _label, _v in (("graph_breaks['n_graph_breaks']", _gb["n_graph_breaks"]),
                   ("graph_breaks['n_graphs']",       _gb["n_graphs"]),
                   ("graph_break_fix['fixed_breaks']", _fx["fixed_breaks"])):
    assert isinstance(_v, int) and _v >= 0, (
        f"{_label} should be a non-negative whole number, got {_v!r}.")
assert _gb["n_graphs"] >= 1, (
    "n_graphs must be at least 1: even a model that breaks everywhere still "
    "compiles at least one subgraph.")
print(f"[OK] counts are whole numbers: {_gb['n_graph_breaks']} breaks, "
      f"{_gb['n_graphs']} subgraphs, {_fx['fixed_breaks']} after the fix")
record("a_break_counts_int")


In [ ]:
# A6 (1 pt): the Part 5 kernel counts must be well-formed.
#
# Deliberately lenient. Whether Inductor dumps artifacts at all, and whether it
# sends the matmuls to cuBLAS or generates them in Triton, both vary by torch
# version. So null is accepted, and zero extern calls is accepted. What is NOT
# accepted is a negative count or a total of zero alongside a claim that the
# counts were collected.
_r = load_results(RESULTS_JSON); _cg = _r["codegen"]
_t, _e = _cg["triton_kernels"], _cg["extern_kernels"]
if _t is None and _e is None:
    print("[OK] kernel counts not collected (accepted - some torch builds "
          "refuse to dump artifacts)")
else:
    assert _t is not None and _e is not None, (
        "Export both codegen.triton_kernels and codegen.extern_kernels, or "
        "neither. One present and one null means Part 5 half-ran.")
    assert _t >= 0 and _e >= 0, "Kernel counts cannot be negative."
    assert _t + _e >= 1, (
        "A compiled transformer block launches at least one kernel, so the "
        "counts cannot both be zero. Re-run Part 5.")
    print(f"[OK] kernel counts well-formed: {_t} Triton + {_e} extern")
record("a_codegen_shape")


---
# Part B — Self-consistency (18 ungraded points)

Each check below recomputes one of your derived values from your own raw
measurements and requires the two to agree within 2%. These relations hold for
any genuine run regardless of hardware — and they are what a hand-edited
results file fails.

In [ ]:
# B1 (4 pts): speedup must equal eager_ms / compiled_ms.
# The headline number of the lab, recomputed from the two latencies it came
# from. Carries the most weight in Part B.
_r = load_results(RESULTS_JSON); _p = _r["perf"]
_exp = _p["eager_ms"] / _p["compiled_ms"]
_e = rel(_p["speedup"], _exp)
assert _e <= TOL, (
    f"speedup = {_p['speedup']:.4f} but eager_ms/compiled_ms = {_exp:.4f}, off "
    f"by {_e*100:.1f}%. Speedup is the eager latency divided by the compiled "
    f"latency - check you did not invert it.")
print(f"[OK] speedup consistent with the two latencies ({_e*100:.2f}% difference)")
record("b_speedup_from_latencies")


In [ ]:
# B2 (2 pts): landscape_row_w3["eager_vs_compiled_speedup"] mirrors perf.
# The row is rounded to 2 decimals on purpose, so compare with a tolerance
# that allows the rounding but not a different number.
_r = load_results(RESULTS_JSON)
_row = num(_r["landscape_row_w3"]["eager_vs_compiled_speedup"],
           "landscape_row_w3['eager_vs_compiled_speedup']")
_act = _r["perf"]["speedup"]
assert abs(_row - _act) <= max(0.01, abs(_act) * TOL), (
    f"landscape_row_w3['eager_vs_compiled_speedup'] = {_row} but your measured "
    f"speedup is {_act:.4f}. The row should hold round(speedup, 2).")
print(f"[OK] landscape row speedup mirrors your measurement ({_row})")
record("b_landscape_speedup")


In [ ]:
# B3 (4 pts): compilation overhead must be first_call minus steady_state.
# The whole point of Part 1b: the first call pays for compilation, every later
# call does not. If compile_cost was not exported this check is skipped with
# full credit, since the field is optional in older copies of the lab.
_r = load_results(RESULTS_JSON); _cc = _r["compile_cost"]
if any(_cc.get(_k) is None for _k in
       ("first_call_ms", "steady_state_ms", "compilation_overhead_ms")):
    print("[OK] compile_cost not fully exported; skipped with credit")
else:
    assert _cc["first_call_ms"] > _cc["steady_state_ms"], (
        f"first_call_ms ({_cc['first_call_ms']:.1f}) should exceed "
        f"steady_state_ms ({_cc['steady_state_ms']:.3f}): the first call is the "
        f"one that pays for compilation.")
    _exp = _cc["first_call_ms"] - _cc["steady_state_ms"]
    _e = rel(_cc["compilation_overhead_ms"], _exp)
    assert _e <= TOL, (
        f"compilation_overhead_ms = {_cc['compilation_overhead_ms']:.3f} but "
        f"first_call_ms - steady_state_ms = {_exp:.3f}, off by {_e*100:.1f}%.")
    print(f"[OK] compilation overhead consistent "
          f"({_cc['compilation_overhead_ms']:.1f} ms of one-time cost)")
record("b_compile_overhead")


In [ ]:
# B4 (2 pts): savings per call must be eager_baseline minus steady_state.
# This one is allowed to be negative - and on a T4 it often is, because the
# compiled block is not reliably faster. A negative saving is an honest result.
_r = load_results(RESULTS_JSON); _cc = _r["compile_cost"]
if any(_cc.get(_k) is None for _k in
       ("eager_baseline_ms", "steady_state_ms", "savings_per_call_ms")):
    print("[OK] compile_cost not fully exported; skipped with credit")
else:
    _exp = _cc["eager_baseline_ms"] - _cc["steady_state_ms"]
    _e = abs(_cc["savings_per_call_ms"] - _exp) / max(abs(_exp), 1e-6)
    assert _e <= 0.05, (
        f"savings_per_call_ms = {_cc['savings_per_call_ms']:.4f} but "
        f"eager_baseline_ms - steady_state_ms = {_exp:.4f}. This is a "
        f"difference of two small numbers, so a 5% tolerance applies.")
    _verdict = ("compiled is faster" if _cc["savings_per_call_ms"] > 0
                else "compiled is NOT faster - an honest result on a small block")
    print(f"[OK] savings per call consistent "
          f"({_cc['savings_per_call_ms']:+.4f} ms/call; {_verdict})")
record("b_savings_per_call")


In [ ]:
# B5 (3 pts): the graph-break slowdown must equal broken_ms / clean_ms.
_r = load_results(RESULTS_JSON); _gb = _r["graph_breaks"]
assert _gb["clean_ms"] > 0 and _gb["broken_ms"] > 0, "Both latencies must be positive."
_exp = _gb["broken_ms"] / _gb["clean_ms"]
_e = rel(_gb["slowdown"], _exp)
assert _e <= TOL, (
    f"slowdown = {_gb['slowdown']:.4f} but broken_ms/clean_ms = {_exp:.4f}, "
    f"off by {_e*100:.1f}%.")
print(f"[OK] break slowdown consistent with the two latencies "
      f"({_gb['slowdown']:.2f}x)")
record("b_slowdown_from_breaks")


In [ ]:
# B6 (2 pts): the fix's speedup must equal broken_ms / fixed_ms.
_r = load_results(RESULTS_JSON); _gb = _r["graph_breaks"]; _fx = _r["graph_break_fix"]
assert _fx["fixed_ms"] > 0, "fixed_ms must be positive."
if _fx["speedup_vs_broken"] is None:
    print("[OK] speedup_vs_broken not exported; fixed_ms positive")
else:
    _exp = _gb["broken_ms"] / _fx["fixed_ms"]
    _e = rel(_fx["speedup_vs_broken"], _exp)
    assert _e <= TOL, (
        f"speedup_vs_broken = {_fx['speedup_vs_broken']:.4f} but "
        f"broken_ms/fixed_ms = {_exp:.4f}, off by {_e*100:.1f}%.")
    print(f"[OK] fix speedup consistent ({_fx['speedup_vs_broken']:.2f}x)")
record("b_fix_speedup")


In [ ]:
# B7 (1 pt): the fixed model should land near the clean model's latency.
#
# Removing the breaks restores whole-graph compilation, so the fixed model and
# the clean model are the same computation compiled the same way.
#
# Only 1 point, and a very wide band, because this comparison is the noisiest
# number in the lab: the two latencies are measured in different cells after
# several intervening compiles, and Dynamo state (recompile limits hit by the
# broken model) can leave the later measurement slower through no fault of the
# student. Prefer clean_ms_recheck, measured in the same cell as fixed_ms.
_r = load_results(RESULTS_JSON); _fx = _r["graph_break_fix"]
_ref = _fx.get("clean_ms_recheck") or _r["graph_breaks"]["clean_ms"]
_which = ("clean_ms_recheck (same-cell)" if _fx.get("clean_ms_recheck")
          else "graph_breaks.clean_ms")
_ratio = _fx["fixed_ms"] / _ref
assert 0.4 <= _ratio <= 2.5, (
    f"fixed_ms = {_fx['fixed_ms']:.3f} ms is {_ratio:.2f}x the clean compiled "
    f"latency ({_ref:.3f} ms, from {_which}). With the breaks removed these "
    f"are the same computation, so they should be in the same ballpark. If the "
    f"gap is large, re-measure after a dynamo.reset().")
if _fx["fix_diff"] is not None:
    assert _fx["fix_diff"] < 1e-3, (
        f"fix_diff = {_fx['fix_diff']:.2e} exceeds 1e-3: the fixed model must "
        f"compute the same function as the original.")
print(f"[OK] fixed model near the clean latency ({_ratio:.2f}x vs {_which}) "
      f"and preserves numerics")
record("b_fixed_matches_clean")


---
# Part C — Physics and plausibility (10 ungraded points)

The graph-break mechanics are **exact** facts about how Dynamo works, so they
are checked as equalities. The timing checks are deliberately wide bands,
generous enough for whatever hardware Colab assigned you and however contended
the VM was.

In [ ]:
# C1 (3 pts): subgraphs = graph breaks + 1. This is exact, not a band.
#
# Each break cuts one traced region into two, so n cuts on a single original
# region give n + 1 regions. The fencepost identity holds on every GPU, every
# model and every torch version.
_r = load_results(RESULTS_JSON); _gb = _r["graph_breaks"]
_b, _g = _gb["n_graph_breaks"], _gb["n_graphs"]
assert _g == _b + 1, (
    f"You reported {_b} graph breaks and {_g} subgraphs, but breaks + 1 = "
    f"{_b + 1}. Each break splits one region in two, so the subgraph count is "
    f"always one more than the break count. Check you read graph_count and "
    f"graph_break_count from the same dynamo.explain() result.")
print(f"[OK] {_b} breaks -> {_g} subgraphs (= breaks + 1, exactly)")
record("c_graphs_equal_breaks_plus_one")


In [ ]:
# C2 (3 pts): the broken model must have breaks; the fixed model must have none.
# This is the actual learning objective of Parts 3 and 4: find them, then
# eliminate them.
_r = load_results(RESULTS_JSON); _gb = _r["graph_breaks"]; _fx = _r["graph_break_fix"]
assert _gb["n_graph_breaks"] >= 2, (
    f"BrokenTransformerBlock has three deliberate graph-breaking operations "
    f"(.item() with a data-dependent branch, print(), and .cpu().numpy()), so "
    f"dynamo.explain() should report at least 2 breaks. You reported "
    f"{_gb['n_graph_breaks']}. Did you explain the BROKEN model, or the clean "
    f"one by mistake?")
assert _fx["fixed_breaks"] == 0, (
    f"The fixed model still reports {_fx['fixed_breaks']} graph break(s). All "
    f"three breaking operations must go: drop the .item() branch (it is dead "
    f"code), remove the print() and its call_count, and remove the "
    f".cpu().numpy() line.")
print(f"[OK] {_gb['n_graph_breaks']} breaks detected, then driven to 0 by your fix")
record("c_breaks_detected_then_fixed")


In [ ]:
# C3 (2 pts): graph breaks should cost time, not save it.
# Each break forces a return to the Python interpreter and a re-entry into
# compiled code. A generous floor of 0.9x allows for timing noise on a run
# where the penalty happens to be small.
_r = load_results(RESULTS_JSON); _gb = _r["graph_breaks"]
assert _gb["slowdown"] >= 0.9, (
    f"slowdown = {_gb['slowdown']:.2f}x says the model with graph breaks was "
    f"meaningfully FASTER than the cleanly-compiled one. Breaks add interpreter "
    f"round-trips; they do not speed anything up. Check that broken_ms and "
    f"clean_ms are not swapped.")
assert _gb["slowdown"] < 100, (
    f"slowdown = {_gb['slowdown']:.1f}x is implausibly large. Did you measure "
    f"the broken model's FIRST call (which includes compilation) instead of "
    f"its steady state?")
print(f"[OK] graph breaks cost time, not save it ({_gb['slowdown']:.2f}x)")
record("c_breaks_cost_time")


In [ ]:
# C4 (1 pt): all latencies in a plausible band for a small transformer block.
# MiniTransformerBlock is d_model=512, seq=64, batch=4 - roughly a millisecond
# on a T4. The band spans three orders of magnitude to cover any GPU and any
# amount of VM contention.
_r = load_results(RESULTS_JSON)
_vals = {
    "perf.eager_ms":          _r["perf"]["eager_ms"],
    "perf.compiled_ms":       _r["perf"]["compiled_ms"],
    "graph_breaks.broken_ms": _r["graph_breaks"]["broken_ms"],
    "graph_break_fix.fixed_ms": _r["graph_break_fix"]["fixed_ms"],
}
for _b, _v in _r["backends"].items():
    _vals[f"backends.{_b}"] = _v["latency_ms"]
for _k, _v in _vals.items():
    assert 0.01 < _v < 1000, (
        f"{_k} = {_v:g} ms is outside the plausible range 0.01-1000 ms for one "
        f"forward pass of this block. A value in the hundreds usually means a "
        f"first (compiling) call was timed instead of the steady state.")
print(f"[OK] all {len(_vals)} latencies plausible "
      f"({min(_vals.values()):.3f}-{max(_vals.values()):.3f} ms)")
record("c_latency_bands")


In [ ]:
# C5 (1 pt): break-even iterations, as an ORDER OF MAGNITUDE only.
#
# break_even = compilation_overhead / savings_per_call is a ratio of two
# DIFFERENCES, and on a small block the denominator is tiny - so the value is
# numerically unstable and can be enormous or infinite. Both are honest. We
# only check it is not negative and not absurdly small, either of which would
# mean the inputs were mixed up.
_r = load_results(RESULTS_JSON); _cc = _r["compile_cost"]
_be = _cc.get("break_even_iters")
if _be is None:
    # Exported as null when compilation never pays off (savings <= 0).
    _sv = _cc.get("savings_per_call_ms")
    assert _sv is None or _sv <= 0.02, (
        "break_even_iters is null, which means compilation never pays off - "
        "but savings_per_call_ms is clearly positive. Those disagree.")
    print("[OK] break-even is 'never': the compiled block is not faster here, "
          "which is a legitimate result on a workload this small")
elif math.isinf(_be):
    print("[OK] break-even is infinite: compilation never pays off here")
else:
    assert _be >= 0, f"break_even_iters = {_be:g} cannot be negative."
    assert _be >= 1, (
        f"break_even_iters = {_be:g} would mean compilation paid for itself "
        f"within a single call, but it cost "
        f"{_cc.get('compilation_overhead_ms', float('nan')):.0f} ms. Check the "
        f"division: overhead / savings_per_call.")
    print(f"[OK] break-even at {_be:,.0f} iterations "
          f"(order of magnitude 1e{math.floor(math.log10(max(_be,1))):.0f})")
record("c_break_even_magnitude")


In [ ]:
# === Your self-assessment score (read-only readout) =======================
# Run this after running every cell above (Kernel -> Restart & Run All).
# It reports your 40 ungraded Self-Assessment points, so you know which parts
# of the lab went well. Nothing here affects your course grade.
try:
    _rows, _earned, _possible = [], {}, {}
    for _gid, _part, _pts in MANIFEST:
        _ok = _gid in _SCORE
        _earned[_part]   = _earned.get(_part, 0)   + (_pts if _ok else 0)
        _possible[_part] = _possible.get(_part, 0) + _pts
        _rows.append((_part, _gid, _pts, _ok))

    _parts  = [p for p in ("A", "B", "C") if p in _possible]
    _auto_e = sum(_earned[p]   for p in _parts)
    _auto_p = sum(_possible[p] for p in _parts)

    print("=" * 68)
    print("  YOUR SELF-ASSESSMENT SCORE (40 ungraded points)")
    print("=" * 68)
    for _p in _parts:
        _bar = "#" * int(round(18 * _earned[_p] / max(_possible[_p], 1)))
        print(f"  {_p} - {PART_NAMES[_p]:26s} {_earned[_p]:3d} / {_possible[_p]:<3d} {_bar}")
    print(f"  {'-' * 64}")
    print(f"  TOTAL                          {_auto_e:3d} / {_auto_p:<3d}")

    # Per-check detail: every graded cell, what it checks, and the points it
    # earned. This is the full breakdown behind the part totals above.
    print("=" * 68)
    print("  DETAIL - every check, in order")
    print("=" * 68)
    _n = {}
    for _p, _g, _pt, _ok in _rows:
        _n[_p] = _n.get(_p, 0) + 1
        _tag   = f"{_p}{_n[_p]}"
        _mark  = "PASS" if _ok else "FAIL"
        _got   = _pt if _ok else 0
        _what  = CHECK_LABELS.get(_g, _g)
        print(f"  {_tag:<4s} {_mark}  {_got}/{_pt}  {_what}")
    print(f"  {'-' * 64}")
    print(f"  TOTAL      {_auto_e:2d}/{_auto_p:<2d}")

    _failed = [(p, g) for p, g, _, ok in _rows if not ok]
    if _failed:
        print("-" * 68)
        print("  Cells not yet passing - scroll up to their error messages:")
        for _p, _g in _failed:
            print(f"    [{_p}] {_g}  -  {CHECK_LABELS.get(_g, '')}")
        print("  Each is independent: fixing one cannot cost you another.")
    else:
        print("-" * 68)
        print("  Every auto-graded check passed.")

    print("=" * 68)
    print("  What to do next")
    print("  1. Kernel -> Restart & Run All, then re-read this table. If you")
    print("     ran cells out of order, the numbers above can be stale.")
    print("  2. These 40 points are UNGRADED self-assessment feedback,")
    print("     auto-graded from your pasted JSON. No human review step.")
    print("  3. Take the Concept Check quiz: the GRADED item for this lab,")
    print("     a separate auto-graded Coursera Assignment worth 10 points")
    print("     (mandatory, 80% to pass).")
    print("=" * 68)
except Exception as _e:
    # A courtesy readout must never be the reason a passing student panics.
    print(f"[score readout unavailable: {type(_e).__name__}: {_e}]")
    print("Your checks above are unaffected - each prints [OK] when it passes.")


---
## What to do next

1. **Run every cell top to bottom** (Kernel → Restart & Run All). Each check
   prints `[OK]` when it passes, and the **score table** just above reports your
   40-point self-assessment total.
2. Any cell that raises an error scores 0 **for that check only** — the checks are
   independent, so a single failure tells you exactly which part of the lab to
   revisit.
3. Confirm the echo table under your pasted JSON matches your Colab output.
4. Take the **Concept Check quiz** — a separate auto-graded Coursera **Graded
   Assignment** worth **10 graded points**. It is mandatory and requires **80%**
   to pass, and it is the only part of this lab that counts toward your course
   grade.

> **These 40 points are feedback, not a grade.** They tell you how well your
> Lab 3 run went. Nothing in this notebook is submitted or affects your course
> grade — the Concept Check quiz is the graded item.

### Troubleshooting

| Symptom | Fix |
|---|---|
| `Could not parse your results` | Paste as plain text; curly "smart quotes" break JSON. Copy from `{` through `}`. |
| `Your pasted JSON looks incomplete` | You copied only part of the file. Select all of it. |
| `Key '...' is missing` | Re-run **Part 7** of the Colab lab. If `backends` is missing you have an older copy of Lab 3 — re-download it. |
| `backends is missing [...]` | The Part 2 ablation loop did not finish all three backends. Re-run it after a `dynamo.reset()`. |
| `speedup = ... but eager_ms/compiled_ms = ...` | The ratio is inverted, or one latency was re-measured after the other. Re-run Part 1. |
| `You reported N breaks and M subgraphs` | Read `graph_break_count` and `graph_count` from the **same** `dynamo.explain()` result. |
| `The fixed model still reports N graph break(s)` | All three breaking ops must go: the `.item()` branch, the `print()` + `call_count`, and the `.cpu().numpy()` line. |
| `fixed_ms ... is N.NNx the clean compiled latency` | Stale Dynamo state. Re-run Part 4 after `dynamo.reset()`. |
| latency `implausibly large` | You timed the first (compiling) call. Warm up, then measure. |

Your Colab notebook stays yours — nothing from it is uploaded, and nothing in
this notebook is collected.

---

*Course: Dataflow Architectures for AI, Week 3 Lab, v1.0 · © 2026 Dr. Kaoutar El Maghraoui*